In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../0_common/01. env-config"

In [0]:
%run "../0_common/03.silver-helpers"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
drivers_df = spark.table(bronze_table).filter((F.col("batch_id") == v_batch_id))
display(drivers_df.head(5))

In [0]:
drivers_df = drivers_df.drop("url")

In [0]:
drivers_df = drivers_df.withColumnsRenamed(
    {
        "driverId":"driver_id",
        "dateOfBirth":"date_of_birth"
    }
)

In [0]:
drivers_df = drivers_df.dropDuplicates(["driver_id"])

In [0]:
import pyspark.sql.functions as F
drivers_df = drivers_df.withColumn(
    "driver_name",
    F.initcap(
        F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))
    )
)

In [0]:
drivers_df = drivers_df.drop("name")

In [0]:
drivers_df = drivers_df.withColumn(
    "nationality",
    F.initcap(F.col("nationality"))
)

In [0]:
drivers_df.dropDuplicates(["driver_id"])

In [0]:
columns_to_update = [col for col in drivers_df.columns if col not in ["driver_id"]]
columns_to_update

In [0]:
write_to_silver(
    df=drivers_df,
    target_table=silver_table,
    columns_to_update=columns_to_update,
    merge_condition="t.driver_id = s.driver_id"
)

In [0]:
display(spark.table(silver_table).limit(5))